In [1]:
import os, gc
from pathlib import Path
import numpy as np

import pandas as pd
from tqdm import tqdm

In [2]:
atlas = "Schaefer2018Combined"

In [3]:
strategies = [
    'wang2023SimpleCorrMatrix', 'wang2023SimpleGSRCorrMatrix'
]

In [4]:
path_pheno = "/scratch/pbergere/COBRE/bids_dataset/participants.tsv"
path_halfpipe = "/scratch/pbergere/halfpipe_massif_COBRE/unzipped_outputs_cobre/derivatives/halfpipe"
path_project = Path("/scratch/pbergere/project_computational_medicine/datasets")

In [5]:
df_pheno = pd.read_csv(path_pheno, sep=None, engine="python")

In [6]:
list_sub = os.listdir(path_halfpipe)

# On garde uniquement les éléments de list_sub présents dans df_pheno["participant_id"]
list_sub = [s for s in list_sub if s in df_pheno["participant_id"].values]

In [7]:
len(list_sub)

148

In [8]:
pheno_idx = df_pheno.set_index("participant_id")[["age", "sex"]]

In [9]:
def upper_triangle_values(mat):
    """Retourne les valeurs de la diagonale supérieure (k=1) d'une matrrice carrée."""
    A = np.asarray(mat)
    if A.shape[0] != A.shape[1]:
        raise ValueError("La matrice de corrélation doit être carrée.")
    iu = np.triu_indices(A.shape[0], k=1)
    return A[iu], iu

def build_path(sub_id, strategy):
    return (
        f"{path_halfpipe}/{sub_id}/func/task-rest/"
        f"{sub_id}_task-rest_feature-{strategy}_atlas-{atlas}_desc-correlation_matrix.tsv"
    )

In [11]:
for strategy in strategies:
    print("\n==============================")
    print(f"Traitement de la stratégie : {strategy}")
    print("==============================")

    rows = []
    corr_cols = None
    missing_files = []

    for sub_id in tqdm(list_sub, desc=f"{strategy}"):
        # phéno pour ce participant (sans session)
        pheno_sub = df_pheno[df_pheno["participant_id"] == sub_id]
        if pheno_sub.empty:
            # pas de métadonnées pour ce sujet, on skip
            continue

        path_corr = build_path(sub_id, strategy)

        if not os.path.exists(path_corr):
            missing_files.append(sub_id)
            continue

        # Lecture
        try:
            df_corr_matrix = pd.read_csv(path_corr, sep="\t", header=None)
        except Exception as e:
            print(f"⚠️ Erreur de lecture pour {sub_id}: {e}")
            missing_files.append(sub_id)
            continue

        # Corrélations (diagonale supérieure)
        A = df_corr_matrix.values
        iu = np.triu_indices(A.shape[0], k=1)
        vals = A[iu]

        # Créer (une seule fois) les noms de colonnes corr_<row>_<col>
        if corr_cols is None:
            corr_cols = [f"corr_{i+1}_{j+1}" for i, j in zip(*iu)]

        # Métadonnées
        # AJOUT ICI : J'ai ajouté "DX_GROUP" pour pouvoir le renommer ensuite
        meta = pheno_sub.iloc[0][["group", "age", "sex"]].to_dict()

        # Construire la ligne
        row = {"participant_id": sub_id, **meta}
        row.update({c: v for c, v in zip(corr_cols, vals)})
        rows.append(row)

        # libérer un peu au fil de l’eau
        del df_corr_matrix, A, vals

    # DataFrame final
    df_out = pd.DataFrame(rows)

    # --- MODIFICATION : Renommage des colonnes ---
    rename_map = {
        "sex": "gender",
    }
    # On utilise errors='ignore' au cas où une colonne manquerait, mais normalement elles sont là
    df_out.rename(columns=rename_map, inplace=True)

    # --- MODIFICATION : Sauvegarde en TSV ---
    out_file = path_project / f"schaefcomb_{strategy}_cobre.tsv"
    
    # sep="\t" crée un TSV
    df_out.to_csv(out_file, sep="\t", index=False)

    print(f"\n✅ Fichier exporté : {out_file}")
    print(f" - sujets sans fichier : {len(missing_files)}")
    if missing_files:
        print(missing_files[:20], "..." if len(missing_files) > 20 else "")

    # Libération mémoire
    del df_out, rows, missing_files, corr_cols
    gc.collect()


Traitement de la stratégie : wang2023SimpleCorrMatrix


wang2023SimpleCorrMatrix: 100%|██████████| 148/148 [00:03<00:00, 44.44it/s]



✅ Fichier exporté : /scratch/pbergere/project_computational_medicine/datasets/schaefcomb_wang2023SimpleCorrMatrix_cobre.tsv
 - sujets sans fichier : 0

Traitement de la stratégie : wang2023SimpleGSRCorrMatrix


wang2023SimpleGSRCorrMatrix: 100%|██████████| 148/148 [00:03<00:00, 42.29it/s]



✅ Fichier exporté : /scratch/pbergere/project_computational_medicine/datasets/schaefcomb_wang2023SimpleGSRCorrMatrix_cobre.tsv
 - sujets sans fichier : 0
